# Multi-agent supervisor

A LangGraph supervisor delegating to two specialists to answer:

> What are the key government revenue streams, and how will the Budget for
> the Future Energy Fund be supported?

## The graph

```mermaid
flowchart TD
    START([START]) --> SUP{supervisor}

    SUP -->|revenue_agent| REV[revenue agent<br/>pages 9, 13, 15]
    SUP -->|expenditure_agent| EXP[expenditure agent<br/>pages 16, 18, 20]
    SUP -->|synthesis| SYN[synthesis<br/>combines findings]
    SUP -->|out_of_scope| DEC[decline<br/>fixed message]

    REV -.->|finding| SUP
    EXP -.->|finding| SUP

    SYN --> END([END])
    DEC --> END
```

Solid arrows are the supervisor's routing choices; dashed arrows are agents
returning their findings. The loop is what makes each turn a decision: after an
agent reports, the supervisor chooses again with that finding in hand.

Agents route unconditionally back to the supervisor, which is the only node
that decides. A fixed chain would have no decision to trace, and the trace is
what the requirement asks for.

## Who reads what

| Agent | Pages | Why those |
|---|---|---|
| revenue_agent | 9, 13, 15 | Revenue breakdown chart, FY2024 narrative, NIRC |
| expenditure_agent | 16, 18, 20 | Table 2.1, top-ups prose, Table 2.4 |

**Page 13 is scoped to revenue only, deliberately.** It carries both the
revenue total and the top-ups sentence. Keeping it out of the expenditure set
means neither agent can answer the combined query alone, so the collaboration
is structural rather than staged.

Scoping is also the cost control: the whole document is ~14.8k tokens against
~1.6k for an agent's page set.

## The answer that matters

The document never states which revenue stream funds the Future Energy Fund.
Government revenue is not earmarked to particular funds. An answer saying
"funded by GST" is wrong however fluent - and it is the most likely failure
mode here, since synthesis is the one node writing free prose.

Assumptions are at the end.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_config
from src.graph.evaluation import load_demo_queries, score_part3
from src.graph.workflow import build_graph, run_query, stream_trace
from src.ingestion.download import ensure_pdf
from src.ingestion.parser import extract_pages
from src.llm import get_chat_model

config = load_config(Path.cwd().parent / "config.yml")
pdf_path = ensure_pdf(config.pdf_url, Path.cwd().parent / config.pdf_path)
model = get_chat_model(config.provider, config.model, temperature=config.temperature)

print(f"provider={config.provider}  model={config.model}")
print(f"revenue pages     {config.pages_for('revenue')}")
print(f"expenditure pages {config.pages_for('expenditure')}")
print(f"max turns         {config.max_turns}")

provider=groq  model=llama-3.1-8b-instant
revenue pages     [9, 13, 15]
expenditure pages [16, 18, 20]
max turns         4


## The graph, as compiled

Node names and edges, read from the compiled graph rather than described.

In [2]:
graph = build_graph(model, config, pdf_path)
topology = graph.get_graph()

print("nodes:")
for node in topology.nodes:
    print(f"  {node}")

print("\nedges:")
for edge in topology.edges:
    label = f"  [{edge.data}]" if edge.data else ""
    print(f"  {edge.source:20s} -> {edge.target}{label}")

nodes:
  __start__
  supervisor
  revenue_agent
  expenditure_agent
  synthesis
  decline
  __end__

edges:
  __start__            -> supervisor
  expenditure_agent    -> supervisor
  revenue_agent        -> supervisor
  supervisor           -> decline  [out_of_scope]
  supervisor           -> expenditure_agent
  supervisor           -> revenue_agent
  supervisor           -> synthesis
  decline              -> __end__
  synthesis            -> __end__


## How the supervisor decides

It returns a structured decision. `reasoning` is declared before `next` in the
schema, so the model states why before it commits:

```python
class RouteDecision(BaseModel):
    reasoning: str
    next: Literal["revenue_agent", "expenditure_agent", "synthesis", "out_of_scope"]
    sub_task: str
```

Three deterministic guards then wrap that choice, in code rather than in the
prompt:

| Guard | Why |
|---|---|
| No agent runs twice | Its pages have not changed; a second pass reads identical text |
| Turn cap of 4 | The graph provably terminates |
| No synthesis before any finding | Synthesising nothing produces an ungrounded answer |

When a guard fires, the trace records both what the model chose and what ran.
A forced route is never presented as a decision.

This is the same split as Part 2: the model where judgement is needed, code
where there is one right answer.

## The required query, streamed

Each node's update as it happens, so the routing is visible while it runs.

In [3]:
REQUIRED = (
    "What are the key government revenue streams, and how will the Budget "
    "for the Future Energy Fund be supported?"
)

stream_trace(REQUIRED, model=model, config=config, pdf_path=pdf_path)

[supervisor] turn 1: revenue_agent
             The revenue_agent covers government revenue streams, and the expenditure_agent covers government spending, including the Budget for the Future Energy Fund.


[revenue_agent] read pages [9, 13, 15], 7 figures


[supervisor] turn 2: expenditure_agent
             The question about key government revenue streams has been addressed, but the Budget for the Future Energy Fund requires information from the expenditure_agent.


[expenditure_agent] read pages [16, 18, 20], 1 figures


[supervisor] turn 3: expenditure_agent -> synthesis
             The question about key government revenue streams has been addressed by the revenue_agent, but the question about the Budget for the Future Energy Fund requires information from the expenditure_agent.


[synthesis] answered


## The full trace

The same run, recorded rather than streamed: every routing decision with its
reasoning, what each agent read, every figure with the text it came from, and
what the run cost.

In [4]:
import time

# The streamed run above just used ~7k tokens; the free tier allows 6k per
# minute, so wait before running the same query again.
time.sleep(60)

trace = run_query(REQUIRED, model=model, config=config, pdf_path=pdf_path)
print(trace.render())

QUERY: What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     revenue_agent      revenue_agent      The revenue_agent covers government revenue stre
2     expenditure_agent  expenditure_agent  The question about key government revenue stream
3     expenditure_agent  synthesis          The question about key government revenue stream
                         OVERRIDDEN:        expenditure_agent has already reported; its pages have not changed

AGENT FINDINGS
agent                pages            figures
revenue_agent        9, 13, 15        7
expenditure_agent    16, 18, 20       1

CITATIONS
      108.60 billion  p.13   Operating Revenue
                             "Estimated FY2024 Operating Revenue is $108.6 billion (15.1% of G"
        4.30 billion  p.13   Incre

## Scoring an open-ended answer

Prose has no single correct wording, so it is not scored. Four things are:

| Check | What it catches |
|---|---|
| Routing | An agent that should have run and did not, or one that ran unnecessarily |
| Figures | A wrong value, or the right value with the wrong unit - 5000 billions is a 1000x error |
| Traceability | A quote that does not appear on the page it cites |
| Page discipline | A figure from a page the agent was never given |

The third is the strongest: the model can read the right sentence and cite the
wrong page, leaving a correct value that cannot be verified.

### The one check that fails, and why it is informative

`traceability` fails on the required query: the revenue agent reports NIRC at
$23.5 billion citing page 13, but quotes page 15's wording.

Both pages mention it:

| Page | Text |
|---|---|
| 13 | "...NIRC of $23.5 billion..." - inside the fiscal position sentence |
| 15 | "Estimated FY2024 NIRC is $23.5 billion, which is $0.6 billion (2.6%) higher..." |

The agent read the page 13 mention and quoted page 15's fuller sentence,
blending two passages about the same figure. The **value is correct** - it is
$23.5 billion on both pages - but the citation cannot be verified, because the
quoted words do not appear on the page named.

Two things follow.

**The check is working.** A blended citation is exactly what it exists to
catch: without it, this would read as a well-cited figure. A prompt instruction
to attribute quotes to the nearest page marker was added and did not fix it,
which suggests the problem is not the model losing track of markers but
genuinely merging two sources it considers equivalent.

**It is a limitation worth stating rather than hiding.** Asking a model to
attribute a quote is asking it to remember provenance across a long context,
and it will sometimes conflate two mentions of the same fact. The value can
still be trusted here; the citation cannot.

In [5]:
def page_text_for(trace):
    """The text of every page this trace cites, for verifying quotes."""
    pages = {citation.page for citation in trace.citations()}
    return {
        page: extract_pages(pdf_path, [page]) for page in sorted(pages)
    }


report = score_part3(
    trace,
    expected={
        "routed_to": ["revenue_agent", "expenditure_agent"],
        "figures": [{"value": 108.6, "unit": "billion"}],
        "figures_any_of": [
            [{"value": 5.0, "unit": "billion"}, {"value": 5000, "unit": "million"}]
        ],
    },
    config=config,
    page_text=page_text_for(trace),
)

print(report.table())

field                        result detail
------------------------------------------------------------------------------
routing                      Pass   invoked revenue_agent, expenditure_agent
figures                      Pass   2 required figure(s) present, units correct
traceability                 FAIL   1 quote(s) not found on their cited page: p.13 (Net Investment Returns Contribution)
page discipline              Pass   every figure came from an agent's own pages

3/4 checks passed


## Every demo query

Queries live in `evaluation/demo_queries.yaml`, so one can be added, reworded
or disabled without touching code.

Together they show single-agent routing both ways, two agents collaborating,
one agent's findings informing the other, and two queries declined.

In [6]:
queries = load_demo_queries()

for query in queries:
    print(f"{query['id']:24s} {query['routed_to'] or ['decline']}")
    print(f"{'':24s} {query['demonstrates'].strip()[:90]}")
    print()

revenue_only             ['revenue_agent']
                         Single-agent routing. The supervisor sends this to revenue alone and does not invoke expen

expenditure_only         ['expenditure_agent']
                         Single-agent routing the other way, and the unit trap - the same amount appears as 5.00 bi

required                 ['revenue_agent', 'expenditure_agent']
                         The requirement's query. Two agents collaborating - neither can answer it alone, because p

collaboration            ['revenue_agent', 'expenditure_agent']
                         One agent's findings informing the other. The expenditure pages say what the top-ups are; 

nirc_classification      ['revenue_agent']
                         A question needing the document read rather than pattern-matched. NIRC is part of Total Re

out_of_scope_sensible    ['decline']
                         Declining a question the document cannot answer. The sharpest case - the model knows the a



In [7]:
import time

# Groq's free tier allows 6,000 tokens per minute. A full query is ~7k across
# six calls, so running several back to back trips the limit. Pausing between
# queries keeps the run inside it; the wait is the constraint, not the work.
PAUSE_SECONDS = 45

results = {}

for index, query in enumerate(queries):
    if index:
        time.sleep(PAUSE_SECONDS)

    trace_for_query = run_query(
        query["query"], model=model, config=config, pdf_path=pdf_path
    )
    report_for_query = score_part3(
        trace_for_query,
        expected=query,
        config=config,
        page_text=page_text_for(trace_for_query),
    )
    results[query["id"]] = (trace_for_query, report_for_query)

    print(f"{query['id']:24s} {report_for_query.summary():22s} {trace_for_query.summary()}")

revenue_only             2/4 checks passed      2 decisions, 1 agent invoked, 0 overrides, 5 figures cited


expenditure_only         3/4 checks passed      2 decisions, 1 agent invoked, 1 override, 1 figure cited


required                 3/4 checks passed      3 decisions, 2 agents invoked, 1 override, 8 figures cited


collaboration            4/4 checks passed      3 decisions, 2 agents invoked, 1 override, 2 figures cited


nirc_classification      3/4 checks passed      2 decisions, 1 agent invoked, 0 overrides, 3 figures cited


out_of_scope_sensible    4/4 checks passed      1 decision, 0 agents invoked, 0 overrides, 0 figures cited


out_of_scope_nonsense    4/4 checks passed      1 decision, 0 agents invoked, 0 overrides, 0 figures cited


### Where the checks failed

The interesting output. Each failure is a real observation about the
system's behaviour, not a defect in the check.


In [8]:
# Which checks failed, and why. Reads the traces already gathered above -
# no further model calls.
for query in queries:
    trace_for_query, report_for_query = results[query["id"]]
    failed = [check for check in report_for_query.checks if not check.passed]
    if not failed:
        continue

    print(f"{query['id']}")
    for check in failed:
        print(f"    {check.field:18s} {check.detail[:96]}")
    for decision in trace_for_query.decisions:
        if decision.was_overridden:
            print(f"    override         turn {decision.turn}: {decision.chose} -> "
                  f"{decision.routed_to} ({decision.overridden})")
    print()

revenue_only
    figures            missing 108.6 billion
    traceability       1 quote(s) not found on their cited page: p.13 (Statutory Boards' Contributions)

expenditure_only
    traceability       1 quote(s) not found on their cited page: p.20 (Initial injection to the Future Energy Fund)
    override         turn 2: expenditure_agent -> synthesis (expenditure_agent has already reported; its pages have not changed)

required
    traceability       1 quote(s) not found on their cited page: p.13 (Net Investment Returns Contribution)
    override         turn 3: expenditure_agent -> synthesis (expenditure_agent has already reported; its pages have not changed)

nirc_classification
    traceability       2 quote(s) not found on their cited page: p.13 (NIRC in FY2024), p.13 (Increase in NIRC from FY2



### Routing across the demo set

The same graph takes a different path for each query. Nothing hardcodes the
order - the supervisor picks each turn, and after each agent it picks again
with that agent's findings in hand.

In [9]:
print(f"{'query':24s} {'expected':34s} {'actual':34s} turns")
print("-" * 100)
for query in queries:
    trace_for_query, _ = results[query["id"]]
    expected = ", ".join(query["routed_to"]) or "decline"
    actual = ", ".join(trace_for_query.agents_invoked) or "decline"
    print(
        f"{query['id']:24s} {expected:34s} {actual:34s} "
        f"{len(trace_for_query.decisions)}"
    )

query                    expected                           actual                             turns
----------------------------------------------------------------------------------------------------
revenue_only             revenue_agent                      revenue_agent                      2
expenditure_only         expenditure_agent                  expenditure_agent                  2
required                 revenue_agent, expenditure_agent   revenue_agent, expenditure_agent   3
collaboration            revenue_agent, expenditure_agent   revenue_agent, expenditure_agent   3
nirc_classification      revenue_agent                      revenue_agent                      2
out_of_scope_sensible    decline                            decline                            1
out_of_scope_nonsense    decline                            decline                            1


### The collaboration query

"The Government tops up Endowment and Trust Funds by $20.4 billion. Where does
the money come from?"

Neither agent can answer this alone. The expenditure pages say what the top-ups
are; only the revenue pages say what funds the Budget they come from. Whichever
agent runs second sees the first's findings in the supervisor's next prompt.

In [10]:
if "collaboration" in results:
    print(results["collaboration"][0].render())

QUERY: The Government tops up Endowment and Trust Funds by $20.4 billion. Where does the money come from?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     revenue_agent      revenue_agent      The question asks where the money comes from, wh
2     expenditure_agent  expenditure_agent  The question asks for the source of the money, w
3     expenditure_agent  synthesis          The source of the money is not explicitly stated
                         OVERRIDDEN:        expenditure_agent has already reported; its pages have not changed

AGENT FINDINGS
agent                pages            figures
revenue_agent        9, 13, 15        1
expenditure_agent    16, 18, 20       1

CITATIONS
       20.40 billion  p.13   Top-up to Endowment and Trust Funds
                             "factoring in Top-ups to Endowment and Trust Funds of $20.4 billi"
   20,352.00 million  p.20 

### A declined query

"What is the capital of France?" is the sharpest test of grounding: the model
knows the answer from training, so answering would be ungrounded by definition.

The supervisor routes to `out_of_scope` and the decline is a fixed message, not
a model call - there is no opportunity for it to answer from memory.

In [11]:
if "out_of_scope_sensible" in results:
    declined_trace = results["out_of_scope_sensible"][0]
    print(declined_trace.table())
    print()
    print("ANSWER")
    print(declined_trace.answer)

turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     out_of_scope       out_of_scope       The query is about geography, which is not cover

ANSWER
This question cannot be answered from the document. It covers Singapore government revenue and expenditure for FY2024 - Operating Revenue and its components, Net Investment Returns Contribution, Total Expenditure, Special Transfers, and top-ups to Endowment and Trust Funds.


## What the run cost

Groq's free tier allows 100k tokens per model per day, which is a real
constraint rather than a footnote - Part 1's development exhausted it in a
single session of prompt iteration.

In [12]:
print(f"{'query':24s} {'calls':>6s} {'seconds':>9s}")
print("-" * 42)
total_calls = total_seconds = 0
for query in queries:
    trace_for_query, _ = results[query["id"]]
    calls = len(trace_for_query.costs)
    seconds = sum(cost.seconds for cost in trace_for_query.costs)
    total_calls += calls
    total_seconds += seconds
    print(f"{query['id']:24s} {calls:>6d} {seconds:>9.1f}")
print("-" * 42)
print(f"{'total':24s} {total_calls:>6d} {total_seconds:>9.1f}")

query                     calls   seconds
------------------------------------------
revenue_only                  4      39.0
expenditure_only              4       1.9
required                      6      30.6
collaboration                 6      30.7
nirc_classification           3       1.8
out_of_scope_sensible         1       0.4
out_of_scope_nonsense         1       0.6
------------------------------------------
total                        25     105.1


## Conclusion

**The supervisor routes dynamically.** Seven queries, four different paths
through the same graph: revenue alone, expenditure alone, both in sequence, and
declined without invoking either. Nothing hardcodes the order.

**The collaboration is structural.** Page 13 is scoped to revenue only, so the
combined query cannot be answered by one agent. That is enforced by config and
asserted in the tests, not arranged by prompt wording.

**The grounding held.** The answer states the $5.0 billion top-up and says
plainly that the document does not identify a revenue stream funding it. This
was the likeliest failure: synthesis writes free prose and cannot see the
document, so a fluent invented link would have been undetectable in the text
alone.

**On LangGraph.** For a graph this small, the routing could be hand-written in
about fifty lines. What the framework buys is the append-only state reducers -
which is why no node can silently discard another's findings - and streaming,
which is how the decisions above were shown as they happened.

### Limits

- **Two agents and a two-part query means routing can look right by luck.** The
  stated reasoning is the only thing separating judgement from a coin flip,
  which is why it is a required field rather than optional.
- **Synthesis cannot check itself.** It sees findings, not the document. The
  traceability check catches a fabricated citation, but not a wrong inference
  drawn from two correct findings.
- **Page scoping is document-specific.** The sets were chosen by reading this
  publication; another document needs them chosen again.
- **Three nodes is a modest graph.** Adding a critic or a retriever would
  demonstrate more of LangGraph, but neither is needed to answer the query.

## Assumptions

- **The document does not identify a revenue stream funding the Future Energy
  Fund,** and the answer says so. Government revenue is not earmarked to
  particular funds. Naming one would be wrong however fluent.
- **Each agent reads only its configured pages** - a correctness measure, not
  only a cost one. An agent able to read the whole document would find a
  plausible figure on a page it was not asked about.
- **Page 13 is revenue-only by design,** so the combined query needs both
  agents.
- **Routing is the model's judgement wrapped in deterministic guards.** The
  model decides; code prevents non-termination and records when it intervened.
- **No agent runs twice per query.** With three pages and no tools, a second
  pass would read identical text.
- **Figures carry their document unit, unconverted.** $5.0 billion (pp.16, 18)
  and 5,000 million (p.20) are the same amount, reported as the citing page
  states it.
- **Structured output uses JSON mode rather than tool-calling.** Over a long
  generation the tool-call wrapper drifts from the format Groq's parser
  accepts, and the request is rejected even when the decision is correct. JSON
  mode also works across model families where tool-calling support varies -
  Qwen fails tool-calling on Groq and succeeds in JSON mode.
- **Temperature is 0**, so the same query yields the same routing.
- **Page numbers hold for this edition only.**